# Ajuste final del NER (B1 + anotación de Doccano) y evaluación ciega

1. Parte el export de Doccano en entrenamiento, validación y prueba (70/15/15, semilla 13, orden independiente del export).
2. Ajusta B1 (AI4Privacy + WikiANN → MEDDOCAN) con el entrenamiento de Doccano junto con MEDDOCAN train, para no olvidar lo aprendido. La mejor época se elige con la validación de Doccano y MEDDOCAN dev.
3. Muestra la validación. **Solo después** corre la prueba de Doccano, una única vez y sin ajustar nada después.

Necesita GPU T4. Si `ner-bert-b1` no está en tu Drive (`MyDrive/modelos-ner/ner-bert-b1`), el paso 2b lo vuelve a entrenar igual que en el experimento (misma semilla, unos 40 minutos) y lo guarda en Drive.

## 1. Subir el código y el export de Doccano
Sube `modelo-ner-v5.zip` y después `andres.jsonl` (el export de Doccano).

In [ ]:
from google.colab import files
import zipfile, os, shutil
subido = files.upload()
zipfile.ZipFile(next(iter(subido))).extractall('/content')
os.chdir('/content/modelo-ner')
export = files.upload()
shutil.move(next(iter(export)), 'datos/doccano/export_doccano.jsonl')
!nvidia-smi --query-gpu=name --format=csv

## 2. Dependencias y modelo B1 desde Drive

In [ ]:
for p in ["accelerate", "presidio-analyzer", "presidio-anonymizer", "datasets"]:
    !pip install -q {p}
!python -m spacy download es_core_news_sm -q
from google.colab import drive; drive.mount('/content/drive')
B1 = '/content/drive/MyDrive/modelos-ner/ner-bert-b1'
print('B1 en Drive:', os.path.exists(B1 + '/config.json'))

## 2b. Reconstruir B1 si no está en Drive
Repite el experimento: AI4Privacy-es + WikiANN-es y después MEDDOCAN, con la misma semilla. Si B1 ya está en Drive, esta celda no hace nada.

In [ ]:
if not os.path.exists(B1 + '/config.json'):
    !python scripts/convertir_ai4privacy.py --n 20000 --n-val 2000 --salida datos/ai4privacy
    !python scripts/convertir_wikiann.py --salida datos/wikiann
    !python scripts/entrenar_ner.py --datos datos/ai4privacy/entrenamiento.jsonl datos/wikiann/entrenamiento.jsonl --val datos/ai4privacy/validacion.jsonl datos/wikiann/validacion.jsonl --base dccuchile/bert-base-spanish-wwm-cased --salida modelos/ner-bert-ai4p-etapa1 --epocas 2 --batch 16
    !python scripts/entrenar_ner.py --datos datos/meddocan/train.jsonl --val datos/meddocan/dev.jsonl --base modelos/ner-bert-ai4p-etapa1 --salida modelos/ner-bert-b1 --epocas 4 --batch 16
    print(open('modelos/ner-bert-b1/reporte_validacion.txt').read())
    !mkdir -p /content/drive/MyDrive/modelos-ner && rsync -a --exclude checkpoints modelos/ner-bert-b1 /content/drive/MyDrive/modelos-ner/
assert os.path.exists(B1 + '/config.json'), 'B1 no quedó en Drive: revisa los errores de la celda.'

## 3. Partición 70/15/15

In [ ]:
!python scripts/particionar_doccano.py --datos datos/doccano/export_doccano.jsonl --salida datos/doccano

## 4. Ajuste final (unos 20 a 30 minutos)

In [ ]:
!python scripts/entrenar_ner.py --datos datos/doccano/entrenamiento.jsonl datos/meddocan/train.jsonl --val datos/doccano/validacion.jsonl datos/meddocan/dev.jsonl --base {B1} --salida modelos/ner-bert-final --epocas 3 --batch 16

## 5. Validación (antes de mirar la prueba)

In [ ]:
print(open('modelos/ner-bert-final/reporte_validacion.txt').read())
!python scripts/evaluar_deteccion.py --datos datos/doccano/validacion.jsonl --nombre doccano_validacion_final --ner-backend hf --ner-modelo modelos/ner-bert-final
!python scripts/evaluar_deteccion.py --datos datos/sintetico/dev.jsonl --nombre sintetico_dev_final --ner-backend hf --ner-modelo modelos/ner-bert-final --sin-presidio

## 6. Evaluación ciega: prueba de Doccano (una sola vez)
También se repiten el sintético test y MEDDOCAN test con el modelo final; esos no son ciegos (sección 8.3).

In [ ]:
!python scripts/evaluar_deteccion.py --datos datos/doccano/prueba.jsonl --nombre doccano_prueba_final --ner-backend hf --ner-modelo modelos/ner-bert-final
!python scripts/evaluar_deteccion.py --datos datos/sintetico/test.jsonl --nombre sintetico_test_final --ner-backend hf --ner-modelo modelos/ner-bert-final
!python scripts/evaluar_deteccion.py --datos datos/meddocan/test.jsonl --nombre meddocan_test_final --ner-backend hf --ner-modelo modelos/ner-bert-final

## 7. Descargar resultados y guardar el modelo

In [ ]:
!zip -qr resultados_final.zip resultados/doccano_validacion_final resultados/sintetico_dev_final resultados/doccano_prueba_final resultados/sintetico_test_final resultados/meddocan_test_final modelos/ner-bert-final/reporte_validacion.txt datos/doccano/entrenamiento.jsonl datos/doccano/validacion.jsonl datos/doccano/prueba.jsonl
files.download('resultados_final.zip')
!rsync -a --exclude checkpoints modelos/ner-bert-final /content/drive/MyDrive/modelos-ner/